The purpose of this template file is to do pre-processing and feature selection.

In [2]:
!pip install braindecode[hub]==1.8.1
import braindecode

/usr/local/lib/python3.13/dist-packages/braindecode/models/eegpt.py:497: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  montage = mne.channels.make_standard_montage("standard_1020")
/usr/local/lib/python3.13/dist-packages/braindecode/models/eegpt.py:1452: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  montage = make_standard_montage("standard_1020")


In [3]:
import os
import importlib
import sys
from pathlib import Path
import json

from tqdm import tqdm

import numpy as np
import matplotlib
from matplotlib import pyplot as plt

from sklearn.manifold import TSNE

import h5py
import cv2

In [4]:
sys.path.append("../src")
import emd_loader
import labram
#import biot
#import cbramod.py
#import classical_features
#import labram_special
#import luna

import helpers
from helpers import save_features, load_features

importlib.reload(emd_loader)
importlib.reload(helpers)

<module 'helpers' from '/content/helpers.py'>

# Loading data

In [5]:
from google.colab import drive
drive.mount("/content/drive/", force_remount=True)
emd_dir = Path("/content/drive/MyDrive/eeg_moments_dataset")

Mounted at /content/drive/


In [ ]:
# The feature extraction is designed this way, because of random crashes of google collab

sub_ids = range(1,6+1)
ses_ids = range(1,3+1)

def compute_individual_features(feature_dir: Path):
    for sub_id in tqdm(sub_ids, desc="Subjects"):
        for ses_id in tqdm(ses_ids, leave=False, desc="Sessions"):
            feature_path = feature_dir / f"feature-{sub_id}-{ses_id}.h5"
            if feature_path.exists():
                continue
            dataset = emd_loader.EmdDataset(emd_dir)
            # X: (n_trials, n_channels, n_times)
            tmp_X, ch_names, tmp_Y = dataset.get_data_by_ids(sub_id, ses_id)
            res = labram.labram_embed(tmp_X, 500, ch_names, batch_size = 32, device = "cuda")

            save_features(feature_path, tmp_X, tmp_Y)

def merge_features(feature_dir: Path):
    X = []
    Y = []
    for sub_id in tqdm(sub_ids, desc="Subjects"):
        for ses_id in tqdm(ses_ids, leave=False, desc="Sessions"):
            feature_path = feature_dir / f"feature-{sub_id}-{ses_id}.h5"
            tmp_X, tmp_Y = load_features(feature_path)
            X.append(tmp_X)
            Y.append(tmp_Y)
    X = np.concatenate(X)
    Y = np.concatenate(Y)
    all_features_path = feature_dir / f"features.h5"
    save_features(all_features_path)

feature_dir = Path("/content/drive/MyDrive/eeg_features/emd_labram")
#feature_dir = Path("/content/drive/MyDrive/eeg_features/emd_biot")
#feature_dir = Path("/content/drive/MyDrive/eeg_features/emd_cbramod")
#feature_dir = Path("/content/drive/MyDrive/eeg_features/emd_labram_special")
#feature_dir = Path("/content/drive/MyDrive/eeg_features/emd_luna")
feature_dir.mkdir(exist_ok=True, parents=True)

print("Compute individual features")
compute_individual_features(feature_dir)

print("Merge features")
merge_features(feature_dir)

Compute individual features


Sessions:   0%|          | 0/8 [00:00<?, ?it/s]
                                               
Sessions:   0%|          | 0/8 [00:00<?, ?it/s]
                                               
Sessions:   0%|          | 0/8 [00:00<?, ?it/s]

Kept 50/127 channels.



Sessions:  12%|█▎        | 1/8 [01:02<07:19, 62.76s/it]

Kept 50/127 channels.


...

Kept 50/127 channels.



Sessions:  75%|███████▌  | 6/8 [07:07<02:21, 70.86s/it]

Kept 50/127 channels.



Sessions:  88%|████████▊ | 7/8 [08:26<01:13, 73.51s/it]

Kept 50/127 channels.


In [ ]:
def tsne_visualize(X, y, title):
    X_2d = TSNE(n_components=2 ,perplexity=30, random_state=42).fit_transform(X)
    classes = np.unique(y)
    for cls in classes:
        idx = y == cls
        plt.scatter(
            X_2d[idx, 0],
            X_2d[idx, 1],
            s=7,
        )
    plt.title(title)
tsne_visualize(res, Y, "LaBraM")